# LeetCode 1382 — Balance a Binary Search Tree bằng Python

Link đề: https://leetcode.com/problems/balance-a-binary-search-tree/ · Độ khó: **Medium** · Chủ đề: Divide and Conquer, Greedy, Tree, Depth-First Search, Binary Search Tree, Binary Tree

Skeleton LeetCode:

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None): ...
class Solution:
    def balanceBST(self, root: TreeNode | None) -> TreeNode | None:
        ...
```

Bài có **nhiều đáp án đúng** ⇒ trong notebook ta kiểm tra kết quả bằng **hàm hợp lệ** (duyệt trung thứ tự giữ nguyên dãy giá trị + cây cân bằng), không so khớp với một cây cố định.

## 1. Mục tiêu học

1. Nắm tính chất nền của **cây nhị phân tìm kiếm (BST)**: duyệt **trung thứ tự (inorder)** cho ra dãy **tăng dần**.
2. Hiểu định nghĩa **cân bằng (height-balanced)**: tại **mọi** nút, chiều cao hai cây con lệch nhau không quá 1.
3. Cài đặt lời giải chuẩn: inorder → mảng đã sắp → **dựng cây cân bằng bằng chia đôi** (chọn phần tử giữa làm gốc), `O(n)`.
4. Viết hàm kiểm tra tính hợp lệ (`inorder` giữ nguyên + cân bằng) trong `O(n)` để test bài nhiều đáp án.
5. (Nâng cao) Biết thuật toán **DSW (Day–Stout–Warren)**: cân bằng tại chỗ bằng phép xoay, bộ nhớ phụ `O(1)`.

## 2. Đề bài & ví dụ

Cho gốc `root` của một BST, trả về một BST **cân bằng** chứa **đúng các giá trị** đó. Nếu có nhiều đáp án, trả về bất kỳ đáp án nào.

Một BST là cân bằng nếu độ sâu của hai cây con của **mọi** nút chênh lệch không quá 1.

- **Ví dụ 1:** `root = [1,null,2,null,3,null,4,null,null]` → `[2,1,3,null,null,null,4]` (cũng đúng: `[3,1,4,null,2]`).
- **Ví dụ 2:** `root = [2,1,3]` → `[2,1,3]`.

**Ràng buộc:** số nút trong `[1, 10^4]`; `1 <= Node.val <= 10^5`.

Lưu ý: cây đầu vào có thể là một "dây" `10^4` nút (Ví dụ 1 là dây 4 nút) ⇒ cẩn thận với đệ quy sâu trong Python.

In [1]:
import random
import sys
import time
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = trống)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q, i = deque([root]), 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

def tree_to_array(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val)
        q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

EXAMPLES = [[1, None, 2, None, 3, None, 4, None, None], [2, 1, 3]]
for arr in EXAMPLES:
    print(arr, "->", tree_to_array(build_tree(arr)))

[1, None, 2, None, 3, None, 4, None, None] -> [1, None, 2, None, 3, None, 4]
[2, 1, 3] -> [2, 1, 3]


## 3. Kiến thức nền

### 3.1 BST và duyệt trung thứ tự

BST: mọi giá trị ở cây con trái `<` nút `<` mọi giá trị ở cây con phải. Duyệt **trái → nút → phải** (inorder) cho dãy **tăng dần**. Bản lặp dùng stack để không lo đệ quy sâu.

In [2]:
def inorder(root):
    out, stack, node = [], [], root
    while stack or node:
        while node:                 # đi hết về trái
            stack.append(node)
            node = node.left
        node = stack.pop()
        out.append(node.val)        # thăm nút
        node = node.right           # sang phải
    return out

print(inorder(build_tree(EXAMPLES[0])))
print(inorder(build_tree([5, 3, 8, 1, 4, 7, 9])))

[1, 2, 3, 4]
[1, 3, 4, 5, 7, 8, 9]


### 3.2 Chiều cao và cân bằng

Chiều cao `h(None) = 0`, `h(node) = 1 + max(h(trái), h(phải))`. Cây **cân bằng** nếu tại **mọi** nút `|h(trái) − h(phải)| ≤ 1`. Kiểm tra hiệu quả: một lượt hậu thứ tự trả về chiều cao, hoặc `-1` nếu đã phát hiện mất cân bằng.

In [3]:
def height(root):
    # lặp theo tầng để an toàn với cây sâu
    h, level = 0, [root] if root else []
    while level:
        h += 1
        level = [c for n in level for c in (n.left, n.right) if c]
    return h

def is_balanced(root):
    """Hậu thứ tự lặp: tính chiều cao từng nút, phát hiện lệch > 1."""
    if root is None:
        return True
    h = {None: 0}
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if done:
            hl, hr = h[node.left], h[node.right]
            if abs(hl - hr) > 1:
                return False
            h[node] = 1 + max(hl, hr)
        else:
            stack.append((node, True))
            if node.right: stack.append((node.right, False))
            if node.left: stack.append((node.left, False))
    return True

for arr in EXAMPLES + [[2, 1, 3, None, None, None, 4]]:
    r = build_tree(arr)
    print(arr, "| chiều cao:", height(r), "| cân bằng:", is_balanced(r))

[1, None, 2, None, 3, None, 4, None, None] | chiều cao: 4 | cân bằng: False
[2, 1, 3] | chiều cao: 2 | cân bằng: True
[2, 1, 3, None, None, None, 4] | chiều cao: 3 | cân bằng: True


### 3.3 Chia để trị trên mảng đã sắp

Với mảng tăng `a[lo..hi]`, chọn `mid = (lo + hi) // 2` làm gốc; nửa trái `a[lo..mid−1]` làm cây con trái, nửa phải `a[mid+1..hi]` làm cây con phải — hai nửa chênh nhau tối đa 1 phần tử. Lặp lại đệ quy. Độ sâu đệ quy chỉ `O(log n)`.

In [4]:
def split_demo(lo, hi, depth=0):
    if lo > hi:
        return
    mid = (lo + hi) // 2
    print("  " * depth + f"[{lo}..{hi}] -> gốc tại {mid}")
    split_demo(lo, mid - 1, depth + 1)
    split_demo(mid + 1, hi, depth + 1)

split_demo(0, 5)

[0..5] -> gốc tại 2
  [0..1] -> gốc tại 0
    [1..1] -> gốc tại 1
  [3..5] -> gốc tại 4
    [3..3] -> gốc tại 3
    [5..5] -> gốc tại 5


### 3.4 Hàm kiểm tra hợp lệ cho bài nhiều đáp án

Một kết quả `res` là đúng khi: (1) `inorder(res) == sorted(giá trị ban đầu)` — đúng các giá trị và vẫn là BST; (2) `is_balanced(res)`. Ta sẽ dùng `check(original_values, res)` cho mọi test.

In [5]:
def check(values_sorted, res):
    return inorder(res) == values_sorted and is_balanced(res)

print(check([1, 2, 3, 4], build_tree([2, 1, 3, None, None, None, 4])))   # đáp án đề
print(check([1, 2, 3, 4], build_tree([3, 1, 4, None, 2])))               # đáp án khác cũng đúng
print(check([1, 2, 3, 4], build_tree([1, None, 2, None, 3, None, 4])))   # không cân bằng

True
True
False


### 3.5 Phép xoay cây (rotation) — cho cách DSW

**Xoay phải** tại `y` có con trái `x`: `x` lên làm gốc, `y` thành con phải của `x`, cây con phải cũ của `x` thành con trái của `y`. Xoay giữ nguyên thứ tự inorder.

In [6]:
def rotate_right(y):
    x = y.left
    y.left = x.right
    x.right = y
    return x

t = build_tree([3, 2, None, 1])          # dây lệch trái 3-2-1
print("trước:", tree_to_array(t), inorder(t))
t = rotate_right(t)
print("sau  :", tree_to_array(t), inorder(t))

trước: [3, 2, None, 1] [1, 2, 3]
sau  : [2, 1, 3] [1, 2, 3]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 là dây `1 → 2 → 3 → 4` (chiều cao 4). Inorder cho `[1,2,3,4]`.

Dựng bằng tay theo chia đôi:
1. Đoạn `[1,2,3,4]`, chỉ số `0..3`, `mid = 1` ⇒ gốc `2`.
2. Trái: `[1]` ⇒ nút lá `1`.
3. Phải: `[3,4]`, `mid` = phần tử đầu ⇒ gốc `3`, con phải `4`.

```
    2
   / \
  1   3
       \
        4
```

Chiều cao giảm từ 4 xuống 3 = `⌈log2(4+1)⌉`. Mọi nút lệch ≤ 1 ⇒ cân bằng. Đây đúng là `[2,1,3,null,null,null,4]` của đề. Nếu chọn `mid = (lo+hi+1)//2` (lệch phải) ta được `[3,2,4,1]` — **cũng đúng**. Vì thế phải test bằng hàm hợp lệ.

## 5. Cách 1 — Brute Force / baseline: gom giá trị, sắp xếp, chèn lần lượt theo thứ tự "giữa trước"

**Ý tưởng:** không tận dụng tính chất BST.
1. Duyệt cây bất kỳ (BFS) gom mọi giá trị rồi `sorted` — `O(n log n)`.
2. Tạo thứ tự chèn "giữa trước" bằng hàng đợi các đoạn (giống BFS trên cây chia đôi).
3. Chèn từng giá trị vào một BST mới bằng thao tác **insert** chuẩn (đi từ gốc xuống) — mỗi lần `O(log n)` vì cây luôn cân bằng theo thứ tự chèn này.

In [7]:
def bst_insert(root, v):
    new = TreeNode(v)
    if root is None:
        return new
    cur = root
    while True:
        if v < cur.val:
            if cur.left is None:
                cur.left = new; break
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = new; break
            cur = cur.right
    return root

def balance_insert(root):
    vals, q = [], deque([root])
    while q:                                   # gom giá trị (thứ tự bất kỳ)
        node = q.popleft()
        if node:
            vals.append(node.val); q.append(node.left); q.append(node.right)
    vals.sort()
    new_root = None
    segs = deque([(0, len(vals) - 1)])
    while segs:                                # thứ tự chèn: giữa trước
        lo, hi = segs.popleft()
        if lo > hi:
            continue
        mid = (lo + hi) // 2
        new_root = bst_insert(new_root, vals[mid])
        segs.append((lo, mid - 1)); segs.append((mid + 1, hi))
    return new_root

for arr in EXAMPLES:
    res = balance_insert(build_tree(arr))
    print(arr, "->", tree_to_array(res), "| hợp lệ:", check(sorted(inorder(build_tree(arr))), res))

[1, None, 2, None, 3, None, 4, None, None] -> [2, 1, 3, None, None, None, 4] | hợp lệ: True
[2, 1, 3] -> [2, 1, 3] | hợp lệ: True


**Độ phức tạp:** sắp xếp `O(n log n)` + `n` lần chèn `O(log n)` ⇒ `O(n log n)`; bộ nhớ `O(n)`.

**Nút thắt:** (1) sắp xếp thừa — BST đã cho sẵn dãy tăng qua inorder trong `O(n)`; (2) mỗi lần chèn lại đi từ gốc xuống `O(log n)`, trong khi ta **biết trước** vị trí của từng phần tử. Ta có thể **dựng trực tiếp** cây từ mảng đã sắp trong `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** inorder của BST = dãy tăng `a`. Mọi BST trên cùng tập giá trị đều có **cùng inorder** — chỉ khác nhau "hình dáng". Vậy bài toán là: *dựng cây có inorder là `a` và chiều cao nhỏ nhất*.

**Insight 2 (chia đôi):** `build(lo, hi)`: gốc = `a[mid]`, `mid = (lo+hi)//2`; cây trái = `build(lo, mid−1)`, cây phải = `build(mid+1, hi)`.
- Hai nửa có kích thước `⌊(k−1)/2⌋` và `⌈(k−1)/2⌉` (chênh ≤ 1).
- **Bất biến:** cây dựng từ `k` phần tử có chiều cao đúng `⌈log2(k+1)⌉` — hàm không giảm theo `k` và hai kích thước chênh 1 cho chiều cao chênh ≤ 1 ⇒ cân bằng.

**Quy tắc "nếu… thì…":** nếu `lo > hi` thì trả `None` (đoạn rỗng); ngược lại tạo/tái dùng nút cho `a[mid]` và **luôn** gán cả `left` lẫn `right`.

**Insight 3 (DSW, nâng cao):** xoay phải liên tục biến cây thành "dây" lệch phải (vine), rồi xoay trái theo từng đợt để "nén" dây thành cây gần hoàn chỉnh — tất cả tại chỗ, `O(n)` thời gian, `O(1)` bộ nhớ phụ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Inorder → mảng giá trị → dựng cây mới bằng chia đôi

In [8]:
def balance_rebuild(root):
    a = inorder(root)                    # dãy tăng, O(n)
    def build(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        node = TreeNode(a[mid])
        node.left = build(lo, mid - 1)
        node.right = build(mid + 1, hi)
        return node
    return build(0, len(a) - 1)

for arr in EXAMPLES:
    res = balance_rebuild(build_tree(arr))
    print(arr, "->", tree_to_array(res), "| hợp lệ:", check(sorted(inorder(build_tree(arr))), res))

[1, None, 2, None, 3, None, 4, None, None] -> [2, 1, 3, None, None, None, 4] | hợp lệ: True
[2, 1, 3] -> [2, 1, 3] | hợp lệ: True


Code nộp LeetCode (Cách 2):

In [9]:
class Solution:
    def balanceBST(self, root: "TreeNode | None") -> "TreeNode | None":
        a, stack, node = [], [], root
        while stack or node:              # inorder lặp: an toàn với dây 10^4 nút
            while node:
                stack.append(node); node = node.left
            node = stack.pop()
            a.append(node.val)
            node = node.right

        def build(lo, hi):
            if lo > hi:
                return None
            mid = (lo + hi) // 2
            return TreeNode(a[mid], build(lo, mid - 1), build(mid + 1, hi))

        return build(0, len(a) - 1)

for arr in EXAMPLES:
    print(tree_to_array(Solution().balanceBST(build_tree(arr))))

[2, 1, 3, None, None, None, 4]
[2, 1, 3]


**Độ phức tạp Cách 2:** inorder `O(n)` + dựng `O(n)` (mỗi phần tử một nút) ⇒ `O(n)`. Bộ nhớ: mảng `O(n)` + cây mới `O(n)` + đệ quy `O(log n)`.

### 7.2 Cách 3 — Inorder → mảng **nút** → nối lại nút cũ (không tạo nút mới)

Giống Cách 2 nhưng lưu **chính các nút** và gán lại `left/right`. Tiết kiệm cấp phát; bắt buộc gán **cả hai** con cho mọi nút (kể cả `None`) để xóa liên kết cũ.

In [10]:
def balance_relink(root):
    nodes, stack, node = [], [], root
    while stack or node:
        while node:
            stack.append(node); node = node.left
        node = stack.pop()
        nodes.append(node)
        node = node.right
    def build(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        nd = nodes[mid]
        nd.left = build(lo, mid - 1)     # luôn gán, kể cả None
        nd.right = build(mid + 1, hi)
        return nd
    return build(0, len(nodes) - 1)

for arr in EXAMPLES:
    vals = sorted(inorder(build_tree(arr)))
    res = balance_relink(build_tree(arr))
    print(arr, "->", tree_to_array(res), "| hợp lệ:", check(vals, res))

[1, None, 2, None, 3, None, 4, None, None] -> [2, 1, 3, None, None, None, 4] | hợp lệ: True
[2, 1, 3] -> [2, 1, 3] | hợp lệ: True


Code nộp LeetCode (Cách 3):

In [11]:
class Solution:
    def balanceBST(self, root: "TreeNode | None") -> "TreeNode | None":
        nodes, stack, node = [], [], root
        while stack or node:
            while node:
                stack.append(node); node = node.left
            node = stack.pop()
            nodes.append(node)
            node = node.right

        def build(lo, hi):
            if lo > hi:
                return None
            mid = (lo + hi) // 2
            nd = nodes[mid]
            nd.left = build(lo, mid - 1)
            nd.right = build(mid + 1, hi)
            return nd

        return build(0, len(nodes) - 1)

for arr in EXAMPLES:
    print(tree_to_array(Solution().balanceBST(build_tree(arr))))

[2, 1, 3, None, None, None, 4]
[2, 1, 3]


**Độ phức tạp Cách 3:** `O(n)` thời gian; bộ nhớ phụ `O(n)` cho mảng nút (không tạo nút mới).

### 7.3 Cách 4 — DSW (Day–Stout–Warren), cân bằng tại chỗ (nâng cao, tùy chọn)

**Từng đoạn:**
1. **Cây → dây (tree-to-vine):** dùng gốc giả `pseudo` có `pseudo.right = root`. Đi xuống theo con phải; gặp nút có con trái thì **xoay phải** tại đó, đến khi mọi nút chỉ có con phải. Đếm `size`.
2. **Dây → cây (vine-to-tree):** gọi `m = 2^⌊log2(size+1)⌋ − 1` (số nút của cây hoàn chỉnh lớn nhất ≤ size). Đầu tiên `compress(size − m)` để xử lý các nút "thừa" ở tầng đáy; sau đó lặp `m //= 2; compress(m)` khi `m > 1`.
3. `compress(count)`: thực hiện `count` lần **xoay trái** cách quãng dọc theo dây.

In [12]:
def compress(pseudo, count):
    scanner = pseudo
    for _ in range(count):
        child = scanner.right
        scanner.right = child.right       # xoay trái tại child
        scanner = scanner.right
        child.right = scanner.left
        scanner.left = child

def balance_dsw(root):
    pseudo = TreeNode(0, None, root)
    # 1) cây -> dây lệch phải
    tail, rest, size = pseudo, root, 0
    while rest:
        if rest.left is None:
            tail, rest = rest, rest.right
            size += 1
        else:
            tmp = rest.left               # xoay phải tại rest
            rest.left = tmp.right
            tmp.right = rest
            rest = tmp
            tail.right = tmp
    # 2) dây -> cây
    m = (1 << (size + 1).bit_length() - 1) - 1
    compress(pseudo, size - m)
    while m > 1:
        m //= 2
        compress(pseudo, m)
    return pseudo.right

for arr in EXAMPLES:
    vals = sorted(inorder(build_tree(arr)))
    res = balance_dsw(build_tree(arr))
    print(arr, "->", tree_to_array(res), "| hợp lệ:", check(vals, res))

[1, None, 2, None, 3, None, 4, None, None] -> [3, 2, 4, 1] | hợp lệ: True
[2, 1, 3] -> [2, 1, 3] | hợp lệ: True


Code nộp LeetCode (Cách 4):

In [13]:
class Solution:
    def balanceBST(self, root: "TreeNode | None") -> "TreeNode | None":
        pseudo = TreeNode(0, None, root)
        tail, rest, size = pseudo, root, 0
        while rest:
            if rest.left is None:
                tail, rest = rest, rest.right
                size += 1
            else:
                tmp = rest.left
                rest.left = tmp.right
                tmp.right = rest
                rest = tmp
                tail.right = tmp

        def compress(count):
            scanner = pseudo
            for _ in range(count):
                child = scanner.right
                scanner.right = child.right
                scanner = scanner.right
                child.right = scanner.left
                scanner.left = child

        m = (1 << (size + 1).bit_length() - 1) - 1
        compress(size - m)
        while m > 1:
            m //= 2
            compress(m)
        return pseudo.right

for arr in EXAMPLES:
    print(tree_to_array(Solution().balanceBST(build_tree(arr))))

[3, 2, 4, 1]
[2, 1, 3]


**Độ phức tạp Cách 4:** mỗi phép xoay `O(1)`; giai đoạn 1 có ≤ `n` phép xoay phải, giai đoạn 2 tổng `≤ n` phép xoay trái ⇒ `O(n)` thời gian, **`O(1)` bộ nhớ phụ** (không mảng, không đệ quy). Kết quả là cây "gần hoàn chỉnh" (mọi tầng đầy trừ tầng đáy) ⇒ chắc chắn cân bằng.

## 8. TRACE từng bước

Trace Cách 2 trên dây 7 nút `1 → 2 → … → 7`. Hàm `trace_build` in mỗi lời gọi `build(lo, hi)`, `mid`, giá trị được chọn làm gốc:

| Lời gọi | mid | gốc | ghi chú |
|---|---|---|---|
| build(0,6) | 3 | 4 | gốc toàn cây |
| build(0,2) | 1 | 2 | cây con trái |
| build(0,0) | 0 | 1 | lá |
| build(2,2) | 2 | 3 | lá |
| build(4,6) | 5 | 6 | cây con phải |
| build(4,4), build(6,6) | 4, 6 | 5, 7 | lá |

In [14]:
def trace_build(arr):
    a = inorder(build_tree(arr))
    print("inorder =", a)
    def build(lo, hi, depth):
        pad = "    " * depth
        if lo > hi:
            print(pad + f"build({lo},{hi}) -> None")
            return None
        mid = (lo + hi) // 2
        print(pad + f"build({lo},{hi}): mid={mid}, gốc={a[mid]}")
        node = TreeNode(a[mid])
        node.left = build(lo, mid - 1, depth + 1)
        node.right = build(mid + 1, hi, depth + 1)
        return node
    res = build(0, len(a) - 1, 0)
    print("kết quả level-order:", tree_to_array(res), "| chiều cao:", height(res))
    return res

chain7 = [1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7]
_ = trace_build(chain7)

inorder = [1, 2, 3, 4, 5, 6, 7]
build(0,6): mid=3, gốc=4
    build(0,2): mid=1, gốc=2
        build(0,0): mid=0, gốc=1
            build(0,-1) -> None
            build(1,0) -> None
        build(2,2): mid=2, gốc=3
            build(2,1) -> None
            build(3,2) -> None
    build(4,6): mid=5, gốc=6
        build(4,4): mid=4, gốc=5
            build(4,3) -> None
            build(5,4) -> None
        build(6,6): mid=6, gốc=7
            build(6,5) -> None
            build(7,6) -> None
kết quả level-order: [4, 2, 6, 1, 3, 5, 7] | chiều cao: 3


Trace Cách 4 (DSW) trên cùng dây 7 nút: dây đã lệch phải nên giai đoạn 1 không xoay; `size = 7`, `m = 7` ⇒ `compress(0)`, rồi `compress(3)`, `compress(1)`. In cây sau mỗi đợt nén:

In [15]:
def trace_dsw(arr):
    root = build_tree(arr)
    pseudo = TreeNode(0, None, root)
    tail, rest, size, rot = pseudo, root, 0, 0
    while rest:
        if rest.left is None:
            tail, rest = rest, rest.right; size += 1
        else:
            tmp = rest.left; rest.left = tmp.right; tmp.right = rest
            rest = tmp; tail.right = tmp; rot += 1
    print(f"giai đoạn 1: {rot} phép xoay phải, size = {size}, dây = {tree_to_array(pseudo.right)}")
    m = (1 << (size + 1).bit_length() - 1) - 1
    print(f"m = {m}; compress({size - m})")
    compress(pseudo, size - m)
    print("   ->", tree_to_array(pseudo.right))
    while m > 1:
        m //= 2
        compress(pseudo, m)
        print(f"compress({m}) ->", tree_to_array(pseudo.right))
    return pseudo.right

res = trace_dsw(chain7)
print("hợp lệ:", check(list(range(1, 8)), res))
print()
res = trace_dsw([5, 3, 8, 1, 4, None, 9, None, 2])   # cây có nhánh trái -> có xoay phải ở giai đoạn 1
print("hợp lệ:", check([1, 2, 3, 4, 5, 8, 9], res))

giai đoạn 1: 0 phép xoay phải, size = 7, dây = [1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7]
m = 7; compress(0)
   -> [1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7]
compress(3) -> [2, 1, 4, None, None, 3, 6, None, None, 5, 7]
compress(1) -> [4, 2, 6, 1, 3, 5, 7]
hợp lệ: True

giai đoạn 1: 4 phép xoay phải, size = 7, dây = [1, None, 2, None, 3, None, 4, None, 5, None, 8, None, 9]
m = 7; compress(0)
   -> [1, None, 2, None, 3, None, 4, None, 5, None, 8, None, 9]
compress(3) -> [2, 1, 4, None, None, 3, 8, None, None, 5, 9]
compress(1) -> [4, 2, 8, 1, 3, 5, 9]
hợp lệ: True


## 9. Code nộp LeetCode

Bản khuyên dùng: Cách 2/3 (inorder lặp + dựng chia đôi). Dưới đây là Cách 3 (tái dùng nút):

In [16]:
class Solution:
    def balanceBST(self, root: "TreeNode | None") -> "TreeNode | None":
        nodes, stack, node = [], [], root
        while stack or node:                    # inorder lặp -> nút theo thứ tự tăng
            while node:
                stack.append(node)
                node = node.left
            node = stack.pop()
            nodes.append(node)
            node = node.right

        def build(lo, hi):                      # dựng cân bằng từ đoạn nodes[lo..hi]
            if lo > hi:
                return None
            mid = (lo + hi) // 2
            nd = nodes[mid]
            nd.left = build(lo, mid - 1)
            nd.right = build(mid + 1, hi)
            return nd

        return build(0, len(nodes) - 1)

for arr in EXAMPLES:
    vals = sorted(inorder(build_tree(arr)))
    res = Solution().balanceBST(build_tree(arr))
    print(tree_to_array(res), "OK" if check(vals, res) else "SAI")

[2, 1, 3, None, None, None, 4] OK
[2, 1, 3] OK


## 10. Vì sao thuật toán đúng

**(1) Vẫn là BST với đúng các giá trị.** `build(lo, hi)` tạo cây có inorder đúng bằng `a[lo..hi]` (quy nạp: inorder = inorder(trái) + `a[mid]` + inorder(phải) = `a[lo..mid−1] + [a[mid]] + a[mid+1..hi]`). Vì `a` tăng dần, cây là BST và chứa đúng các giá trị ban đầu.

**(2) Cân bằng.** Gọi `H(k)` là chiều cao cây dựng từ `k` phần tử. Chứng minh quy nạp `H(k) = ⌈log2(k+1)⌉`: hai cây con có kích thước `⌊(k−1)/2⌋` và `⌈(k−1)/2⌉`, và `H` không giảm, nên chiều cao hai con chênh nhau tối đa `H(⌈(k−1)/2⌉) − H(⌊(k−1)/2⌋) ≤ 1` (hai số chênh 1 có `⌈log2(·+1)⌉` chênh ≤ 1). Áp dụng tại mọi nút ⇒ cân bằng.

**(3) DSW.** Mỗi phép xoay giữ inorder ⇒ vẫn là BST. Sau giai đoạn 2, cây có các tầng `1..⌊log2(size+1)⌋` đầy đủ và tầng cuối chứa `size − m` nút ⇒ cây gần hoàn chỉnh, chiều cao tối thiểu ⇒ cân bằng.

## 11. Phân tích Time / Space

Gọi `n` = số nút (≤ `10^4`), `H` = chiều cao cây ban đầu.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Gom + sort + chèn | `O(n log n)` | `O(n)` |
| 2. Inorder + dựng mới | `O(n)` | `O(n)` mảng + `O(n)` nút mới + `O(H)` stack inorder |
| 3. Inorder + nối lại nút | `O(n)` | `O(n)` mảng nút + `O(H)` |
| 4. DSW | `O(n)` | `O(1)` |

Đệ quy trong `build` chỉ sâu `O(log n)` ≈ 14 tầng với `n = 10^4`; phần nguy hiểm là inorder **đệ quy** trên dây `10^4` nút — vì vậy ta dùng inorder **lặp**.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case (kiểm tra bằng hàm hợp lệ)

In [17]:
METHODS = {
    "insert": balance_insert,
    "rebuild": balance_rebuild,
    "relink": balance_relink,
    "dsw": balance_dsw,
    "submit": lambda r: Solution().balanceBST(r),
}
edge = [
    [1],                                         # 1 nút
    [2, 1],                                      # 2 nút lệch trái
    [1, None, 2],                                # 2 nút lệch phải
    [3, 2, None, 1],                             # dây lệch trái
    [4, 2, 6, 1, 3, 5, 7],                       # đã cân bằng hoàn hảo
    [1, None, 15, 14, None, 13, None, 12],      # zigzag
    [10, 5, 20, None, None, 15, 30, None, None, None, 40, None, 50],
]
for name, fn in METHODS.items():
    for arr in EXAMPLES + edge:
        vals = sorted(inorder(build_tree(arr)))
        res = fn(build_tree(arr))
        assert check(vals, res), (name, arr, tree_to_array(res))
print("Tất cả", len(METHODS), "cách cho kết quả HỢP LỆ trên", len(EXAMPLES) + len(edge), "test")
print("Ví dụ 1 qua các cách:", {k: tree_to_array(f(build_tree(EXAMPLES[0]))) for k, f in METHODS.items()})

Tất cả 5 cách cho kết quả HỢP LỆ trên 9 test
Ví dụ 1 qua các cách: {'insert': [2, 1, 3, None, None, None, 4], 'rebuild': [2, 1, 3, None, None, None, 4], 'relink': [2, 1, 3, None, None, None, 4], 'dsw': [3, 2, 4, 1], 'submit': [2, 1, 3, None, None, None, 4]}


### 12.2 Ngẫu nhiên (seed cố định): BST từ hoán vị ngẫu nhiên các giá trị phân biệt

In [18]:
def random_bst(rng, n):
    vals = rng.sample(range(1, 100_001), n)
    root = None
    for v in vals:
        root = bst_insert(root, v)
    return root

rng = random.Random(1382)
for _ in range(400):
    n = rng.randint(1, 80)
    seed = rng.randrange(10**9)
    for name, fn in METHODS.items():
        root = random_bst(random.Random(seed), n)       # mỗi cách nhận cây mới (Cách 3, 4 sửa tại chỗ)
        vals = inorder(root)
        res = fn(root)
        assert check(vals, res), name
        assert height(res) == n.bit_length()          # chiều cao tối thiểu ceil(log2(n+1))
print("400 BST ngẫu nhiên × 5 cách: hợp lệ và đạt chiều cao tối thiểu")

400 BST ngẫu nhiên × 5 cách: hợp lệ và đạt chiều cao tối thiểu


Ghi chú: `n.bit_length()` = `⌈log2(n+1)⌉` — chiều cao nhỏ nhất của mọi cây nhị phân `n` nút. Cả năm cách đều đạt đúng chiều cao tối thiểu này.

### 12.3 Test lớn theo ràng buộc: dây `10^4` nút

In [19]:
def chain(n, right=True):
    root = TreeNode(1); cur = root
    for v in range(2, n + 1):
        nd = TreeNode(v)
        if right: cur.right = nd; cur = nd
        else:
            nd.left = root; root = nd                 # dây lệch trái: n, n-1, ..., 1
    return root

for name in ["rebuild", "relink", "dsw", "submit"]:
    t0 = time.perf_counter()
    r1 = METHODS[name](chain(10_000, True))
    r2 = METHODS[name](chain(10_000, False))
    ok = check(list(range(1, 10_001)), r1) and check(list(range(1, 10_001)), r2)
    print(f"{name:8s} -> hợp lệ {ok}, chiều cao {height(r1)} | {time.perf_counter() - t0:.3f}s")

rebuild  -> hợp lệ True, chiều cao 14 | 0.024s


relink   -> hợp lệ True, chiều cao 14 | 0.019s
dsw      -> hợp lệ True, chiều cao 14 | 0.013s
submit   -> hợp lệ True, chiều cao 14 | 0.018s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dựng từ thứ tự duyệt **không phải inorder** (ví dụ BFS/preorder) mà không sắp xếp

In [20]:
def wrong_bfs_order(root):
    vals, q = [], deque([root])
    while q:
        nd = q.popleft()
        if nd:
            vals.append(nd.val); q.append(nd.left); q.append(nd.right)
    # SAI: vals theo level-order, KHÔNG tăng dần
    def build(lo, hi):
        if lo > hi: return None
        mid = (lo + hi) // 2
        return TreeNode(vals[mid], build(lo, mid - 1), build(mid + 1, hi))
    return build(0, len(vals) - 1)

arr = [4, 2, 6, 1, 3, 5, 7]
res = wrong_bfs_order(build_tree(arr))
print("inorder kết quả:", inorder(res), "| hợp lệ:", check(list(range(1, 8)), res))

inorder kết quả: [4, 2, 6, 1, 3, 5, 7] | hợp lệ: False


**Vì sao sai:** chia đôi chỉ tạo BST khi mảng **đã sắp tăng**. Thứ tự BFS `[4,2,6,1,3,5,7]` không tăng ⇒ cây cân bằng nhưng **không còn là BST** (inorder `[4,2,6,1,3,5,7]`).

**Code đúng:** dùng inorder (hoặc `sorted`).

### Lỗi 2 — Điều kiện dừng sai (`lo >= hi`) làm mất nút

In [21]:
def wrong_stop(root):
    a = inorder(root)
    def build(lo, hi):
        if lo >= hi:                  # SAI: đoạn 1 phần tử (lo == hi) bị bỏ
            return None
        mid = (lo + hi) // 2
        return TreeNode(a[mid], build(lo, mid - 1), build(mid + 1, hi))
    return build(0, len(a) - 1)

res = wrong_stop(build_tree(EXAMPLES[0]))
print("inorder kết quả:", inorder(res), "| mong đợi [1, 2, 3, 4] | hợp lệ:", check([1, 2, 3, 4], res))

inorder kết quả: [2, 3] | mong đợi [1, 2, 3, 4] | hợp lệ: False


**Vì sao sai:** với đoạn đóng `[lo, hi]`, đoạn rỗng là `lo > hi`; `lo == hi` là đoạn **có 1 phần tử** (một lá). Dừng ở `lo >= hi` làm mất mọi lá ⇒ thiếu giá trị (ở đây mất 1 và 4).

**Code đúng:** `if lo > hi: return None` (đoạn đóng) hoặc `if lo == hi` với đoạn nửa mở `[lo, hi)` — chọn một quy ước và giữ nhất quán.

### Lỗi 3 — Tái dùng nút cũ nhưng chỉ gán con khi đoạn khác rỗng (giữ liên kết cũ)

In [22]:
def wrong_relink(root):
    nodes, stack, node = [], [], root
    while stack or node:
        while node:
            stack.append(node); node = node.left
        node = stack.pop(); nodes.append(node); node = node.right
    def build(lo, hi):
        mid = (lo + hi) // 2
        nd = nodes[mid]
        if lo <= mid - 1:             # SAI: lá giữ nguyên left/right CŨ
            nd.left = build(lo, mid - 1)
        if mid + 1 <= hi:
            nd.right = build(mid + 1, hi)
        return nd
    return build(0, len(nodes) - 1)

def safe_count(root, limit=100):
    """Đếm nút, dừng nếu phát hiện thăm lại (chu trình) để không lặp vô hạn."""
    seen, stack = set(), [root]
    while stack:
        nd = stack.pop()
        if nd is None: continue
        if id(nd) in seen or len(seen) > limit:
            return f"phát hiện chu trình / thăm lại nút {nd.val}"
        seen.add(id(nd)); stack += [nd.left, nd.right]
    return f"{len(seen)} nút"

res = wrong_relink(build_tree(EXAMPLES[0]))
print("wrong_relink:", safe_count(res))
print("đúng        :", safe_count(balance_relink(build_tree(EXAMPLES[0]))))

wrong_relink: phát hiện chu trình / thăm lại nút 2
đúng        : 4 nút


**Vì sao sai:** ở dây `1 → 2 → 3 → 4`, nút `1` là lá trong cây mới nhưng vẫn giữ `right = 2` cũ; `2` lại là gốc ⇒ **chu trình** `2 → 1 → 2 → ...`. Trên LeetCode, trình chấm serialize cây sẽ lặp vô hạn hoặc báo lỗi.

**Code đúng:** luôn gán `nd.left = build(...)` và `nd.right = build(...)`, kể cả khi kết quả là `None`.

### Lỗi 4 — Kiểm tra cân bằng chỉ ở **gốc**

In [23]:
def wrong_is_balanced(root):
    return abs(height(root.left) - height(root.right)) <= 1      # SAI: chỉ xét gốc

t = build_tree([4, 3, 5, 2, None, None, 6, 1, None, None, 7])   # hai dây dài 3 hai bên
print("wrong_is_balanced:", wrong_is_balanced(t), "| is_balanced đúng:", is_balanced(t))

wrong_is_balanced: True | is_balanced đúng: False


**Vì sao sai:** định nghĩa yêu cầu **mọi** nút. Cây trên có gốc với hai nhánh cùng chiều cao 3, nhưng nút `3` có cây con trái cao 2 và cây con phải rỗng ⇒ lệch 2. Hàm sai kết luận "cân bằng" — nếu dùng hàm này để test, ta sẽ bỏ sót lỗi.

**Code đúng:** hậu thứ tự kiểm tra tại từng nút (`is_balanced` ở mục 3.2).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **BST + "dựng lại / chuyển đổi"** ⇒ gần như luôn bắt đầu bằng **inorder** (dãy tăng).
- **"Cây cân bằng từ dãy đã sắp"** ⇒ chia đôi, `mid = (lo+hi)//2` (LeetCode 108, 109).
- Bài "trả về bất kỳ đáp án nào" ⇒ viết **hàm kiểm tra hợp lệ** thay vì so khớp đáp án mẫu.
- Python + cây `10^4` tầng ⇒ inorder **lặp**; phần chia đôi thì đệ quy thoải mái (`log n` tầng).
- Tái dùng nút: **gán lại cả hai con** cho mọi nút.
- Follow-up "bộ nhớ `O(1)`" ⇒ nhắc tới **DSW** (xoay cây).

## 15. Biến thể của bài

1. **LeetCode 108 — Convert Sorted Array to BST:** chính là bước 2 của bài này.
2. **LeetCode 109 — Convert Sorted List to BST:** danh sách liên kết; dựng `O(n)` bằng "inorder mô phỏng" với con trỏ chạy dọc danh sách.
3. **LeetCode 110 — Balanced Binary Tree:** chỉ kiểm tra cân bằng (hàm `is_balanced`).
4. **Cân bằng thành cây hoàn chỉnh (complete)**: DSW cho đúng dạng này; hoặc chọn `mid` để tầng cuối dồn trái.
5. **Cây tự cân bằng (AVL, Red-Black):** giữ cân bằng sau **mỗi** lần chèn/xóa bằng phép xoay — thay vì dựng lại toàn bộ.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Gom + sort + chèn | không dùng tính chất BST; chèn "giữa trước" | `O(n log n)` | `O(n)` | khi cây vào không phải BST |
| 2. Inorder + dựng mới | dãy tăng → chia đôi tạo nút mới | `O(n)` | `O(n)` | **dễ viết nhất**, phỏng vấn |
| 3. Inorder + nối lại nút | như 2 nhưng tái dùng nút | `O(n)` | `O(n)` | tiết kiệm cấp phát |
| 4. DSW | xoay thành dây rồi nén | `O(n)` | `O(1)` | follow-up bộ nhớ hằng |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `tree_to_array(balance_rebuild(...))` cho dây `1 → 2 → … → 6` là gì? (dùng `mid = (lo+hi)//2`)

**Bài 2 (tính tay):** chiều cao tối thiểu của cây cân bằng có `n = 1, 2, 3, 7, 8, 10000` nút là bao nhiêu?

**Bài 3 (điền TODO):** hoàn thiện `sorted_array_to_bst(nums)` (LeetCode 108).

In [24]:
def sorted_array_to_bst(nums):
    """Bài 3 — điền TODO. (Bài tập — chưa gọi)"""
    def build(lo, hi):
        # TODO: đoạn rỗng -> None; mid = (lo+hi)//2; tạo nút nums[mid]; gán hai con đệ quy
        pass
    return build(0, len(nums) - 1)

**Bài 4 (tự viết lại):** viết `is_balanced_rec(root)` bằng đệ quy hậu thứ tự trả về chiều cao, hoặc `-1` nếu cây con đã mất cân bằng (dừng sớm) — `O(n)`.

**Bài 5 (mini challenge — LeetCode 109):** cho danh sách liên kết **đã sắp** (dùng lớp `ListNode` tự dựng), dựng BST cân bằng trong `O(n)` **không chuyển sang mảng**: đếm độ dài, rồi dựng theo kiểu "inorder mô phỏng" với một con trỏ tiến dọc danh sách.

In [25]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def sorted_list_to_bst(head):
    """Bài 5 — tự viết. (Bài tập — chưa gọi)"""
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `build(0,5)`: mid = 2 ⇒ gốc 3 | trái `[1,2]`: gốc 1, con phải 2; phải `[4,5,6]`: gốc 5 | `[3,1,5,None,2,4,6]` |
| 2 | `⌈log2(n+1)⌉` | `n.bit_length()` | 1, 2, 2, 3, 4, 14 |
| 3 | Giống Cách 2 | `if lo > hi: return None` | `TreeNode(nums[mid], build(lo, mid-1), build(mid+1, hi))` |
| 4 | trả chiều cao của cây con | nếu con trả `-1` hoặc lệch > 1 ⇒ trả `-1` | `return check(root) != -1` |
| 5 | Dựng `build(lo, hi)` nhưng **không** dùng chỉ số để lấy giá trị | dựng cây trái trước, rồi lấy `cur.val`, tiến `cur = cur.next`, rồi dựng cây phải | dùng `nonlocal cur` |

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
chain6 = [1, None, 2, None, 3, None, 4, None, 5, None, 6]
ans1 = tree_to_array(balance_rebuild(build_tree(chain6)))
print("Bài 1:", ans1)
assert ans1 == [3, 1, 5, None, 2, 4, 6]

# Bài 2
ans2 = [n.bit_length() for n in (1, 2, 3, 7, 8, 10000)]
print("Bài 2:", ans2)
assert ans2 == [1, 2, 2, 3, 4, 14]
assert height(balance_rebuild(chain(10000))) == 14

Bài 1: [3, 1, 5, None, 2, 4, 6]
Bài 2: [1, 2, 2, 3, 4, 14]


In [27]:
# Bài 3
def sorted_array_to_bst_sol(nums):
    def build(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        return TreeNode(nums[mid], build(lo, mid - 1), build(mid + 1, hi))
    return build(0, len(nums) - 1)

for nums in ([-10, -3, 0, 5, 9], [1, 3], list(range(100))):
    assert check(nums, sorted_array_to_bst_sol(nums))
print("Bài 3: OK")

Bài 3: OK


In [28]:
# Bài 4
def is_balanced_rec(root):
    def h(node):
        if node is None:
            return 0
        hl = h(node.left)
        if hl == -1: return -1
        hr = h(node.right)
        if hr == -1 or abs(hl - hr) > 1: return -1
        return 1 + max(hl, hr)
    return h(root) != -1

rng = random.Random(4)
for _ in range(300):
    r = random_bst(rng, rng.randint(1, 40))
    assert is_balanced_rec(r) == is_balanced(r)
assert is_balanced_rec(build_tree([4, 3, 5, 2, None, None, 6, 1, None, None, 7])) is False
print("Bài 4: OK")

Bài 4: OK


In [29]:
# Bài 5
def sorted_list_to_bst_sol(head):
    n, p = 0, head
    while p:
        n += 1; p = p.next
    cur = head
    def build(lo, hi):
        nonlocal cur
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        left = build(lo, mid - 1)          # dựng trái trước (inorder)
        node = TreeNode(cur.val)
        cur = cur.next
        node.left = left
        node.right = build(mid + 1, hi)
        return node
    return build(0, n - 1)

def make_list(arr):
    head = None
    for v in reversed(arr):
        head = ListNode(v, head)
    return head

for arr in ([-10, -3, 0, 5, 9], [1], [1, 2], list(range(1, 51))):
    assert check(arr, sorted_list_to_bst_sol(make_list(arr)))
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
nodes, stack, node = [], [], root
while stack or node:                    # inorder lặp
    while node: stack.append(node); node = node.left
    node = stack.pop(); nodes.append(node); node = node.right

def build(lo, hi):                      # đoạn đóng [lo, hi]
    if lo > hi: return None
    mid = (lo + hi) // 2
    nd = nodes[mid]
    nd.left, nd.right = build(lo, mid - 1), build(mid + 1, hi)   # LUÔN gán cả hai
    return nd
return build(0, len(nodes) - 1)
```

**Checklist trước khi submit:**
- [ ] Dùng **inorder** (dãy tăng), không dùng BFS/preorder.
- [ ] Điều kiện dừng `lo > hi` khớp quy ước đoạn đóng.
- [ ] Tái dùng nút ⇒ gán lại **cả** `left` và `right`.
- [ ] Inorder **lặp** để chịu được dây `10^4` nút.
- [ ] Test bằng hàm hợp lệ: `inorder(res) == sorted(vals)` và cân bằng tại **mọi** nút.
- [ ] Đúng chữ ký: `balanceBST(self, root) -> TreeNode`.